# Ward and stability analysis
Compare 2 methods under a protocol and test their sensitivity to the sample.

Ward's method iteratively merges groups to minimize the increase in within-cluster variance. It uses Euclidean distances and is sensitive to feature scaling and extreme observations. Its memory requirements grow roughly quadratically with the number of rows, so we limit the size of the comparison.

Here, Ward and GMM use the same training rows, D features, and scaling. These results are a separate training benchmark. They do not replace the time-based validation in Notebook 03.

We do not generate predictions for 2020 or 2021 using Ward: scikit-learn's Ward model does not provide a built-in predict method.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "src/config.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "src/config.py").exists():
    raise FileNotFoundError("Öppna notebooken från paketets notebooks-mapp eller projektrot")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display
from src.config import ProjectConfig, options
from src.io_utils import prepare_run, read_json, require_files, ensure_unlocked, save_figure
from src.data import load_daily_until
from src.modeling import benchmark_ward_vs_gmm, evaluate_clustering_stability

cfg = ProjectConfig()
RUN = prepare_run(ROOT, cfg)
ensure_unlocked(RUN)
FEATURES = ROOT / "output" / "02_time_split_and_preprocessing"
OUT = ROOT / "output" / "04_ward_comparison"
DEV_DATA = ROOT / cfg.development_features_path
for folder in ("models", "tables", "figures"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
opt = options(cfg)
print(f"Experiment: {OUT.name} | Profile: {cfg.run_profile}\nNotebook results: {OUT}")


Experiment: 04_ward_comparison | Profile: full
Notebook results: C:\Users\MY NGOC\Documents\MAU_TACAS_68803\DA631E_ArtificialIntelligenceForDataScience\Project_2_UnsupervisedLearning\output\04_ward_comparison


In [ ]:
ensure_unlocked(OUT)
require_files(OUT, ["candidate_ids.json", "feature_sets.json"])
require_files(DEV_DATA, ['development_features.parquet'])
train = pd.read_pickle(DEV_DATA / "development_features.parquet")
feature_sets = read_json(OUT / "feature_sets.json")
ids = read_json(OUT / "candidate_ids.json")
artifacts = {cid: joblib.load(OUT / "models" / f"{cid}.joblib") for cid in ids.values()}


FileNotFoundError: Kör tidigare notebooks först. Saknas: train_rows.pkl, candidate_ids.json, feature_sets.json

In [ ]:
benchmark, ward_info = ward_benchmark(train, feature_sets["D"], cfg,
                                    covariance=artifacts[ids["D"]]["model"].covariance_type)
benchmark.to_csv(OUT / "ward_benchmark.csv", index=False)
display(benchmark)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

ward = AgglomerativeClustering(n_clusters=opt["k"][0], linkage="ward", metric="euclidean")
ward_labels = ward.fit_predict(ward_info['scaled_features'])
print("First 5 assignments:", ward_labels[:5])
print("Does the model have a built-in method for new rows", hasattr(ward, "predict"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sns.lineplot(data=benchmark, x="k", y="silhouette", hue="method", marker="o", ax=axes[0])
sns.lineplot(data=benchmark, x="k", y="davies_bouldin", hue="method", marker="o", ax=axes[1])
axes[0].set_title("Silhouette: higher is better")
axes[1].set_title("Davies–Bouldin: lower is better")
save_figure(fig, OUT, "04_same_sample_benchmark.png")
plt.show()

## View part of the hierarchy
This dendogram shows how Ward clustering groups the observations in the capped, scaled feature set D. Ward starts with each observation as its own group, then repeatedly merges the pair of groups that causes the smallest increase in within-group variation. The branching feature shows the order of those mergers. The height of a merge indicates how different the merged groups were under this procedure.

The plot is truncated so the large hierarchy is easier to read. As a result, a leaf at the bottom may stand for several observations rather than one stock-day row. The dendogram describes similarities in the selected features. It does not show returns, predict future outcomes, or prove that the groups are naturally distinct.

The observations are not arranged as a time series in this plot. In particular, the left-to-right leaf order should not be interpreted as chronological order. 

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
link = linkage(ward_info["x"], method="ward")
fig, ax = plt.subplots(figsize=(10, 4))
dendrogram(link, truncate_mode="lastp", p=18, ax=ax)
ax.set_title("Ward: trunkerat dendrogram för dagliga träningsobservationer")
ax.set_ylabel("Sammanslagningsavstånd")
save_figure(fig, OUT, "04_ward_dendrogram.png")
plt.show()

## Stability analysis using time blocks
We remove approximately 20% of the consecutive data blocks and refit the models.

For GMM, we compare the cluster assignments on the same unseen 2020 anchor rows using the Adjusted Rand Index (ARI). For Ward, we compare assignments only on the training rows shared by the original and refited samples. These protocols measure stability in different ways, so their ARI scores should not be ranked as if they were identical tests.

ARI is unchanged if cluster labels are swapped. A score of 1 means the 2 partitions match exactly, while a score around 0 indicates similarity close to what would be expected by chance. ARI does not measure economic relevance. Removing date blocks is a sensitivity analysis.

In [ ]:
rows, _ = load_daily_until(ROOT, cfg, cfg.valid_end)
anchor = rows.loc[rows.Date.between(cfg.valid_start, cfg.valid_end)]
stability = evaluate_clustering_stability(train, anchor, artifacts, ward_info, cfg)
stability.to_csv(OUT / "stability.csv", index=False)
display(stability.groupby(["method", "candidate_id", "protocol"]).agg(
    mean_ari=("ari", "mean"), min_ari=("ari", "min"), repeats=("ari", "size"),
    all_converged=("converged", "all")))